# 83 — Simulate10Next: Conqueror & Supplier Tests

Test notebook for `StrategyPipeline` from `82-Simulate10Next_Conqueror_Supplier.py`.
Each test exposes `df_s`, `pa`, `safe`, and `action` for interactive inspection.

In [1]:
%run 82-Simulate10Next_Conqueror_Supplier.py

In [2]:
import copy, math, random
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
# import kaggle_environments as ke

_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}


class Obs:
    def __init__(self, planets, initial_planets=None, fleets=None,
                 next_fleet_id=100, comets=None, comet_planet_ids=None,
                 angular_velocity=0.0):
        self.planets          = [list(p) for p in planets]
        self.initial_planets  = [list(p) for p in (initial_planets if initial_planets is not None else planets)]
        self.fleets           = [list(f) for f in (fleets or [])]
        self.next_fleet_id    = next_fleet_id
        self.comets           = comets or []
        self.comet_planet_ids = comet_planet_ids or []
        self.angular_velocity = angular_velocity


def simulate_with_action(obs, action0, n_steps, current_step=0):
    snapshots = []
    for i, step in enumerate(range(current_step, current_step + n_steps)):
        snapshots.append({
            'step':    step,
            'planets': [p[:] for p in obs.planets],
            'fleets':  [f[:] for f in obs.fleets],
        })
        interpreter(obs, [action0 if i == 0 else [], []], step)
    return snapshots


def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')

    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100)
        ax.set_ylim(100, 0)
        ax.set_aspect('equal')
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y,   str(ships),         ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y+2, str(pid),            ha='center', va='center', color='red',   fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y-2, "+"+str(production), ha='center', va='center', color='white', fontsize=5, fontweight='bold', zorder=4)
        for f in snap['fleets']:
            fid, owner, x, y, angle, from_id, ships = f
            c = _COLORS.get(owner, '#888888')
            ax.plot(x, y, 'D', color=c, markersize=5, zorder=5)
            ax.text(x + 1.5, y + 1.5, str(ships), color=c, fontsize=5, zorder=6)
        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

## Test 01 — 1 Supplier, 1 Conqueror, 1 Enemy

Planet 1 (x=30) should attack planet 2 (x=70). Planet 0 (x=20) stays as supplier.

In [3]:
obs01 = Obs(
    planets=[
        [0, 0, 10.0, 10.0, 1 + math.log(3), 50, 3],  # Supplier  (dist≈56.6, not orbiting)
        [1, 0, 25.0,  5.0, 1 + math.log(3), 50, 3],  # Conqueror (dist≈51.5, not orbiting)
        [2, 1, 75.0,  5.0, 1.0,              1,  1],  # Enemy     (dist≈51.5, not orbiting)
    ],
    angular_velocity=0.05,
)
df_s01, pd01 = StrategyPipeline._01_get_obs_dataframe(obs01, step=0, num_agents=2)
df_s01

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,10.000000,10.000000,2.098612,50,3,0,fix
1,0,1,25.000000,10.000000,2.098612,50,3,0,moving
2,0,2,75.000000,10.000000,1.000000,1,1,1,moving
3,1,0,10.000000,10.000000,2.098612,53,3,0,fix
4,1,1,25.000000,10.000000,2.098612,53,3,0,moving
5,1,2,75.000000,10.000000,1.000000,2,1,1,moving
6,2,0,10.000000,10.000000,2.098612,56,3,0,fix
7,2,1,27.030410,8.800510,2.098612,56,3,0,moving
8,2,2,76.967923,11.299469,1.000000,3,1,1,moving
9,3,0,10.000000,10.000000,2.098612,59,3,0,fix


In [4]:
pa01 = StrategyPipeline._02_get_all_opportunities(df_s01, pd01, player_id=0)
pa01

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,1,0,25.0,10.0,2.098612,50,3,moving,0,11,...,3.141593,15.000000,15.000000,12.901388,15.268909,0.000000,0.137636,3.003957,3.279228,3.141593
1,1,0,25.0,10.0,2.098612,50,3,moving,0,11,...,3.141593,15.000000,15.000000,12.901388,15.325047,0.000000,0.136852,3.004740,3.278445,3.141593
2,1,0,25.0,10.0,2.098612,50,3,moving,0,11,...,3.141593,15.000000,15.000000,12.901388,15.211931,0.000000,0.138330,3.003263,3.279922,3.141593
3,1,0,25.0,10.0,2.098612,50,3,moving,0,11,...,3.141593,15.000000,15.000000,12.901388,15.154084,0.000000,0.138930,3.002662,3.280523,3.141593
4,1,0,25.0,10.0,2.098612,50,3,moving,0,11,...,3.141593,15.000000,15.000000,12.901388,14.975045,0.000000,0.140129,3.001464,3.281721,3.141593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-0.195039,33.675598,35.561689,31.579996,34.139694,0.003446,0.044299,6.043847,6.132446,-0.196762
266,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-0.195039,34.076359,35.561689,31.979614,33.985653,0.002682,0.039863,6.048283,6.128010,-0.196380
267,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-0.195039,34.500219,35.561689,32.402562,33.829050,0.001893,0.034141,6.054005,6.122288,-0.195985
268,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-0.195039,34.949373,35.561689,32.851078,33.669795,0.001078,0.026249,6.061898,6.114395,-0.195578


In [5]:
safe01 = StrategyPipeline._03_filter_collision(pa01)
safe01

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,1,0,25.0,10.0,2.098612,50,3,moving,0,11,...,15.000000,15.000000,12.901388,15.268909,0.000000,0.137636,3.003957,3.279228,3.141593,3.141593
1,1,0,25.0,10.0,2.098612,50,3,moving,0,11,...,15.000000,15.000000,12.901388,15.325047,0.000000,0.136852,3.004740,3.278445,3.141593,3.141593
2,1,0,25.0,10.0,2.098612,50,3,moving,0,11,...,15.000000,15.000000,12.901388,15.211931,0.000000,0.138330,3.003263,3.279922,3.141593,3.141593
3,1,0,25.0,10.0,2.098612,50,3,moving,0,11,...,15.000000,15.000000,12.901388,15.154084,0.000000,0.138930,3.002662,3.280523,3.141593,3.141593
4,1,0,25.0,10.0,2.098612,50,3,moving,0,11,...,15.000000,15.000000,12.901388,14.975045,0.000000,0.140129,3.001464,3.281721,3.141593,3.141593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,33.675598,35.561689,31.579996,34.139694,0.003446,0.044299,6.043847,6.132446,-0.196762,-0.196762
266,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,34.076359,35.561689,31.979614,33.985653,0.002682,0.039863,6.048283,6.128010,-0.196380,-0.196380
267,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,34.500219,35.561689,32.402562,33.829050,0.001893,0.034141,6.054005,6.122288,-0.195985,-0.195985
268,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,34.949373,35.561689,32.851078,33.669795,0.001078,0.026249,6.061898,6.114395,-0.195578,-0.195578


In [6]:
action01 = StrategyPipeline._04_score_and_decide(safe01, player_id=0)
print("Action:", action01)
snaps01 = simulate_with_action(copy.deepcopy(obs01), action01, 20)
make_animation(snaps01, title='Test 01 — 1 Supplier, 1 Conqueror, 1 Enemy', interval=200)

Action: []


## Test 02 — 1 Supplier, 2 Conquerors (one more in need)

Planet 1 attacks planet 2 (easy). Planet 3 attacks planet 4 (heavy — 100 ships). Planet 0 stays as supplier.

In [7]:
obs02 = Obs(
    planets=[
        [0, 0, 10.0, 10.0, 1 + math.log(3), 50,  3],  # dist≈56.6, not orbiting
        [1, 0, 25.0,  5.0, 1 + math.log(3), 50,  3],  # dist≈51.5, not orbiting
        [2, 1, 75.0,  5.0, 1 + math.log(3), 1,   1],  # dist≈51.5, not orbiting
        [3, 0,  5.0, 25.0, 1 + math.log(3), 50,  3],  # dist≈51.5, not orbiting
        [4, 1,  5.0, 75.0, 1 + math.log(3), 100, 1],  # dist≈51.5, not orbiting
    ],
    angular_velocity=0.05,
)
df_s02, pd02 = StrategyPipeline._01_get_obs_dataframe(obs02, step=0, num_agents=2)
df_s02

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,10.000000,10.000000,2.098612,50,3,0,fix
1,0,1,25.000000,10.000000,2.098612,50,3,0,moving
2,0,2,75.000000,10.000000,2.098612,1,1,1,moving
3,0,3,10.000000,25.000000,2.098612,50,3,0,moving
4,0,4,10.000000,75.000000,2.098612,100,1,1,moving
5,1,0,10.000000,10.000000,2.098612,53,3,0,fix
6,1,1,25.000000,10.000000,2.098612,53,3,0,moving
7,1,2,75.000000,10.000000,2.098612,2,1,1,moving
8,1,3,10.000000,25.000000,2.098612,53,3,0,moving
9,1,4,10.000000,75.000000,2.098612,101,1,1,moving


In [8]:
pa02 = StrategyPipeline._02_get_all_opportunities(df_s02, pd02, player_id=0)
pa02

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,1.333206,11.460401,11.453307,9.361803,9.372970,0.000735,0.026673,1.306532,1.359879,1.333573
1,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,1.333206,12.344713,11.453307,10.463735,12.359692,0.081907,0.159254,1.173951,1.497019,1.374159
2,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,1.333206,12.353213,11.453307,10.476601,12.396889,0.082600,0.157477,1.175729,1.498406,1.374506
3,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,1.333206,12.361560,11.453307,10.489284,12.433613,0.083279,0.155652,1.177554,1.499764,1.374845
4,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,1.333206,12.369757,11.453307,10.501791,12.469876,0.083945,0.153778,1.179427,1.501096,1.375178
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-0.195039,33.675598,35.561689,31.579996,34.139694,0.003446,0.044299,6.043847,6.132446,-0.196762
831,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-0.195039,34.076359,35.561689,31.979614,33.985653,0.002682,0.039863,6.048283,6.128010,-0.196380
832,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-0.195039,34.500219,35.561689,32.402562,33.829050,0.001893,0.034141,6.054005,6.122288,-0.195985
833,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-0.195039,34.949373,35.561689,32.851078,33.669795,0.001078,0.026249,6.061898,6.114395,-0.195578


In [9]:
safe02 = StrategyPipeline._03_filter_collision(pa02)
safe02

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,11.460401,11.453307,9.361803,9.372970,0.000735,0.026673,1.306532,1.359879,1.333573,1.333573
1,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,12.344713,11.453307,10.463735,12.359692,0.081907,0.159254,1.173951,1.497019,1.374159,1.374159
2,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,12.353213,11.453307,10.476601,12.396889,0.082600,0.157477,1.175729,1.498406,1.374506,1.374506
3,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,12.361560,11.453307,10.489284,12.433613,0.083279,0.155652,1.177554,1.499764,1.374845,1.374845
4,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,12.369757,11.453307,10.501791,12.469876,0.083945,0.153778,1.179427,1.501096,1.375178,1.375178
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,33.675598,35.561689,31.579996,34.139694,0.003446,0.044299,6.043847,6.132446,-0.196762,-0.196762
831,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,34.076359,35.561689,31.979614,33.985653,0.002682,0.039863,6.048283,6.128010,-0.196380,-0.196380
832,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,34.500219,35.561689,32.402562,33.829050,0.001893,0.034141,6.054005,6.122288,-0.195985,-0.195985
833,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,34.949373,35.561689,32.851078,33.669795,0.001078,0.026249,6.061898,6.114395,-0.195578,-0.195578


In [10]:
action02 = StrategyPipeline._04_score_and_decide(safe02, player_id=0)
print("Action:", action02)
snaps02 = simulate_with_action(copy.deepcopy(obs02), action02, 20)
make_animation(snaps02, title='Test 02 — 1 Supplier, 2 Conquerors (one more in need)', interval=200)

Action: []


## Test 03 — 4 Suppliers, 2 Conquerors (one more in need)

Same enemies as Test 02. Planets 5, 6, 7 added as extra suppliers. Pipeline should still route correctly.

In [11]:
obs03 = Obs(
    planets=[
        [0, 0, 10.0, 10.0, 1 + math.log(3), 50,  3],  # dist≈56.6, not orbiting
        [1, 0, 25.0,  5.0, 1 + math.log(3), 50,  3],  # dist≈51.5, not orbiting
        [2, 1, 75.0,  5.0, 1 + math.log(3), 1,   1],  # dist≈51.5, not orbiting
        [3, 0,  5.0, 25.0, 1 + math.log(3), 50,  3],  # dist≈51.5, not orbiting
        [4, 1,  5.0, 75.0, 1 + math.log(3), 100, 1],  # dist≈51.5, not orbiting
        [5, 0,  5.0,  5.0, 1 + math.log(3), 50,  3],  # dist≈63.6, not orbiting
        [6, 0,  5.0, 15.0, 1 + math.log(3), 50,  3],  # dist≈57.0, not orbiting
        [7, 0, 15.0,  5.0, 1 + math.log(3), 50,  3],  # dist≈57.0, not orbiting
    ],
    angular_velocity=0.05,
)
df_s03, pd03 = StrategyPipeline._01_get_obs_dataframe(obs03, step=0, num_agents=2)
df_s03

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,10.000000,10.000000,2.098612,50,3,0,fix
1,0,1,25.000000,10.000000,2.098612,50,3,0,moving
2,0,2,75.000000,10.000000,2.098612,1,1,1,moving
3,0,3,10.000000,25.000000,2.098612,50,3,0,moving
4,0,4,10.000000,75.000000,2.098612,100,1,1,moving
...,...,...,...,...,...,...,...,...,...
83,10,3,24.856254,10.090201,2.098612,80,3,0,moving
84,10,4,3.107978,55.112556,2.098612,110,1,1,moving
85,10,5,5.000000,5.000000,2.098612,80,3,0,fix
86,10,6,5.000000,15.000000,2.098612,80,3,0,fix


In [12]:
pa03 = StrategyPipeline._02_get_all_opportunities(df_s03, pd03, player_id=0)
pa03

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.279174,0.000000,0.179031,3.747960,4.106022,-2.356194
1,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.261766,0.000000,0.174541,3.752450,4.101532,-2.356194
2,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.244026,0.000000,0.169763,3.757227,4.096754,-2.356194
3,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.225940,0.000000,0.164663,3.762328,4.091654,-2.356194
4,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.207493,0.000000,0.159198,3.767793,4.086189,-2.356194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5361,3,0,10.0,25.0,2.098612,50,3,moving,0,11,...,-1.815775,20.615528,20.615528,20.907561,22.714140,0.100143,0.000000,4.367268,4.567553,-1.815775
5362,7,0,15.0,5.0,2.098612,50,3,fix,0,11,...,2.356194,14.142136,14.142136,15.244281,16.240748,0.121707,0.000000,2.234488,2.477901,2.356194
5363,3,0,10.0,25.0,2.098612,50,3,moving,0,11,...,-1.325818,20.615528,20.615528,20.402100,22.424709,0.101842,0.049468,4.855525,5.059210,-1.325818
5364,3,0,10.0,25.0,2.098612,50,3,moving,0,11,...,-2.034444,11.180340,11.180340,11.198612,12.198612,0.187821,0.157292,4.060920,4.436563,-2.034444


In [13]:
safe03 = StrategyPipeline._03_filter_collision(pa03)
safe03

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.279174,0.000000,1.790308e-01,3.747960,4.106022,-2.356194,-2.356194
1,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.261766,0.000000,1.745412e-01,3.752450,4.101532,-2.356194,-2.356194
2,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.244026,0.000000,1.697634e-01,3.757227,4.096754,-2.356194,-2.356194
3,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.225940,0.000000,1.646629e-01,3.762328,4.091654,-2.356194,-2.356194
4,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.207493,0.000000,1.591983e-01,3.767793,4.086189,-2.356194,-2.356194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4817,3,0,10.0,25.0,2.098612,50,3,moving,0,11,...,20.615528,20.615528,21.380429,22.714140,0.093118,1.490116e-08,4.374293,4.560528,-1.815775,-1.815775
4818,3,0,10.0,25.0,2.098612,50,3,moving,0,11,...,20.615528,20.615528,20.907561,22.714140,0.100143,0.000000e+00,4.367268,4.567553,-1.815775,-1.815775
4819,3,0,10.0,25.0,2.098612,50,3,moving,0,11,...,20.615528,20.615528,20.402100,22.424709,0.101842,4.946774e-02,4.855525,5.059210,-1.325818,-1.325818
4820,3,0,10.0,25.0,2.098612,50,3,moving,0,11,...,11.180340,11.180340,11.198612,12.198612,0.187821,1.572916e-01,4.060920,4.436563,-2.034444,-2.034444


In [14]:
action03 = StrategyPipeline._04_score_and_decide(safe03, player_id=0)
print("Action:", action03)
snaps03 = simulate_with_action(copy.deepcopy(obs03), action03, 20)
make_animation(snaps03, title='Test 03 — 4 Suppliers, 2 Conquerors (one more in need)', interval=200)

Action: []


## Test 04 — 4 Suppliers, 1 Conqueror Orbiting

Planets 4 and 5 orbit (dist_from_center < 50 − radius). `angular_velocity=0.05`. The pipeline must intercept the moving enemy planet.

In [15]:
obs04 = Obs(
    planets=[
        [0, 0, 10.0, 10.0, 1 + math.log(3), 50, 3],  # dist≈56.6, not orbiting
        [1, 0,  5.0,  5.0, 1 + math.log(3), 50, 3],  # dist≈63.6, not orbiting
        [2, 0,  5.0, 15.0, 1 + math.log(3), 50, 3],  # dist≈57.0, not orbiting
        [3, 0, 15.0,  5.0, 1 + math.log(3), 50, 3],  # dist≈57.0, not orbiting
        [4, 0, 20.0, 20.0, 1.0,              50, 1],  # dist≈42.4 < 49, orbiting
        [5, 1, 25.0, 25.0, 1.0,              50, 1],  # dist≈35.4 < 49, orbiting enemy
    ],
    angular_velocity=0.05,
)
df_s04, pd04 = StrategyPipeline._01_get_obs_dataframe(obs04, step=0, num_agents=2)
df_s04

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,10.000000,10.000000,2.098612,50,3,0,fix
1,0,1,5.000000,5.000000,2.098612,50,3,0,fix
2,0,2,5.000000,15.000000,2.098612,50,3,0,fix
3,0,3,15.000000,5.000000,2.098612,50,3,0,fix
4,0,4,15.000000,15.000000,1.000000,50,1,0,fix
...,...,...,...,...,...,...,...,...,...
61,10,1,5.000000,5.000000,2.098612,80,3,0,fix
62,10,2,5.000000,15.000000,2.098612,80,3,0,fix
63,10,3,15.000000,5.000000,2.098612,80,3,0,fix
64,10,4,15.000000,15.000000,1.000000,60,1,0,fix


In [16]:
pa04 = StrategyPipeline._02_get_all_opportunities(df_s04, pd04, player_id=0)
pa04

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.244026,0.000000,0.169763,3.757227,4.096754,-2.356194
1,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.225940,0.000000,0.164663,3.762328,4.091654,-2.356194
2,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.207493,0.000000,0.159198,3.767793,4.086189,-2.356194
3,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.188671,0.000000,0.153319,3.773672,4.080310,-2.356194
4,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,-2.356194,7.071068,7.071068,4.972456,5.169457,0.000000,0.146961,3.780030,4.073952,-2.356194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4127,4,0,15.0,15.0,1.000000,50,1,fix,0,11,...,-1.570796,10.000000,10.000000,10.100000,11.100000,0.208962,0.169840,4.503426,4.921351,-1.570796
4128,4,0,15.0,15.0,1.000000,50,1,fix,0,11,...,3.141593,10.000000,10.000000,11.530360,12.098612,0.133834,0.000000,3.007759,3.275426,3.141593
4129,4,0,15.0,15.0,1.000000,50,1,fix,0,11,...,3.141593,10.000000,10.000000,10.100000,11.100000,0.208962,0.169840,2.932630,3.350555,3.141593
4130,4,0,15.0,15.0,1.000000,50,1,fix,0,11,...,-2.356194,14.142136,14.142136,16.044655,16.240748,0.058811,0.000000,3.868179,3.985802,-2.356194


In [17]:
safe04 = StrategyPipeline._03_filter_collision(pa04)
safe04

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.244026,0.000000,0.169763,3.757227,4.096754,-2.356194,-2.356194
1,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.225940,0.000000,0.164663,3.762328,4.091654,-2.356194,-2.356194
2,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.207493,0.000000,0.159198,3.767793,4.086189,-2.356194,-2.356194
3,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.188671,0.000000,0.153319,3.773672,4.080310,-2.356194,-2.356194
4,0,0,10.0,10.0,2.098612,50,3,fix,0,11,...,7.071068,7.071068,4.972456,5.169457,0.000000,0.146961,3.780030,4.073952,-2.356194,-2.356194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3433,4,0,15.0,15.0,1.000000,50,1,fix,0,11,...,22.389639,23.418692,21.503834,22.906953,0.021149,0.037096,0.031908,0.111302,0.079578,0.079578
3434,4,0,15.0,15.0,1.000000,50,1,fix,0,11,...,10.000000,10.000000,11.530360,12.098612,0.133834,0.000000,4.578555,4.846223,-1.570796,-1.570796
3435,4,0,15.0,15.0,1.000000,50,1,fix,0,11,...,10.000000,10.000000,10.100000,11.100000,0.208962,0.169840,4.503426,4.921351,-1.570796,-1.570796
3436,4,0,15.0,15.0,1.000000,50,1,fix,0,11,...,10.000000,10.000000,11.530360,12.098612,0.133834,0.000000,3.007759,3.275426,3.141593,3.141593


In [18]:
action04 = StrategyPipeline._04_score_and_decide(safe04, player_id=0)
print("Action:", action04)
snaps04 = simulate_with_action(copy.deepcopy(obs04), action04, 20)
make_animation(snaps04, title='Test 04 — 4 Suppliers, 1 Conqueror Orbiting', interval=200)

Action: []


## Test 05 — Our Agent vs Random Agent

Full game via `kaggle_environments`. Player 0 uses `agent()` from `82-...py`, player 1 uses a random policy.

In [19]:
def random_agent_fn(obs):
    player = obs.player
    my_planets = [p for p in obs.planets if p[1] == player]
    if not my_planets:
        return []
    planet = random.choice(my_planets)
    ships = planet[5] // 2
    if ships < 1:
        return []
    return [[planet[0], random.uniform(0, 2 * math.pi), ships]]


# Reset agent globals so this cell is re-runnable
step = 0
num_agents = None
player_id = None

SEED = 42
N_STEPS = 100
random.seed(SEED)

env = ke.make("orbit_wars", debug=False)
env.reset(2)

snaps05 = []
for env_step in range(N_STEPS):
    obs0 = env.state[0].observation
    obs1 = env.state[1].observation
    snaps05.append({
        'step':    env_step,
        'planets': [list(p) for p in obs0.planets],
        'fleets':  [list(f) for f in obs0.fleets],
    })
    action0 = agent(obs0)
    action1 = random_agent_fn(obs1)
    env.step([action0, action1])
    if env.state[0].status != "ACTIVE":
        break

obs0 = env.state[0].observation
snaps05.append({
    'step':    len(snaps05),
    'planets': [list(p) for p in obs0.planets],
    'fleets':  [list(f) for f in obs0.fleets],
})
p0 = sum(p[5] for p in obs0.planets if p[1] == 0)
p1 = sum(p[5] for p in obs0.planets if p[1] == 1)
winner = "Our agent wins" if p0 > p1 else "Random wins" if p1 > p0 else "Tie"
print(f"After {len(snaps05) - 1} steps: {winner}  (player0={p0}, player1={p1})")

make_animation(snaps05, title='Test 05 — Our Agent vs Random', interval=100)

NameError: name 'ke' is not defined